<a href="https://colab.research.google.com/github/YonaUz/RAG-and-LLM/blob/main/01_document_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -U langchain-core langchain-community pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 580.5/580.5 kB 7.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 25.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.7 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from pathlib import Path
from datetime import date
from typing import Any

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, TextLoader, UnstructuredMarkdownLoader

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Satellite_Intelligence_Assistant"
)

DATA_DIR = PROJECT_ROOT / "data"

RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

LANDSAT_DIR = RAW_DATA_DIR / "landsat"
SENTINEL2_DIR = RAW_DATA_DIR / "sentinel2"

for directory in [
    LANDSAT_DIR,
    SENTINEL2_DIR,
    PROCESSED_DATA_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Processed data directory: {PROCESSED_DATA_DIR}")

Project root: /content/drive/MyDrive/Satellite_Intelligence_Assistant
Raw data directory: /content/drive/MyDrive/Satellite_Intelligence_Assistant/data/raw
Processed data directory: /content/drive/MyDrive/Satellite_Intelligence_Assistant/data/processed


In [ ]:
for path in PROJECT_ROOT.rglob("*"):
    print(path.relative_to(PROJECT_ROOT))

data
data/raw
data/processed
data/raw/landsat
data/raw/sentinel2


In [ ]:
DOCUMENT_REGISTRY = {
    "landsat_8_data_users_handbook.pdf": {
        "title": "Landsat 8 Data Users Handbook",
        "publisher": "USGS",
        "mission": "Landsat 8",
        "document_type": "technical_handbook",
        "url": "https://www.usgs.gov/media/files/landsat-8-data-users-handbook",
        "language": "en",
    },
    "landsat_9_data_users_handbook.pdf": {
        "title": "Landsat 9 Data Users Handbook",
        "publisher": "USGS",
        "mission": "Landsat 9",
        "document_type": "technical_handbook",
        "url": "https://www.usgs.gov/landsat-missions/landsat-data-user-handbooks",
        "language": "en",
    },
    "sentinel_2_documentation.pdf": {
        "title": "Sentinel-2 Technical Documentation",
        "publisher": "Copernicus Data Space Ecosystem",
        "mission": "Sentinel-2",
        "document_type": "technical_documentation",
        "url": "https://documentation.dataspace.copernicus.eu/Data/Sentinel2.html",
        "language": "en",
    },
}

In [ ]:
def load_document(file_path: Path) -> list[Document]:
    """Load a PDF or text file into LangChain Document objects."""
    suffix = file_path.suffix.lower()
    if suffix == ".pdf":
        loader = PyPDFLoader(str(file_path))
        documents = loader.load()
    elif suffix == ".md":
        loader = UnstructuredMarkdownLoader(str(file_path))
        documents = loader.load()
    elif suffix == ".txt":
        loader = TextLoader(str(file_path), encoding="utf-8",)
        documents = loader.load()
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}"
        )
    return documents

In [ ]:
landsat_8_path = (
    LANDSAT_DIR / "landsat_8_data_users_handbook.pdf"
)
if landsat_8_path.exists():
    landsat_8_documents = load_document(landsat_8_path)

    print(f"Number of extracted pages: {len(landsat_8_documents)}")
    print("\nFirst page preview:")
    print(landsat_8_documents[0].page_content[:1000])
else:
    print(f"File not found: {landsat_8_path}")

Number of extracted pages: 114

First page preview:
LSDS-1574 
Version 5.0 
Department of the Interior 
U.S. Geological Survey 
 
 
 
 
 
Landsat 8 (L8) 
Data Users Handbook 
 
 
 
 
 
 
 
 
 
 
 
 
 
Version 5.0 
 
November 2019


In [ ]:
def enrich_metadata(
    documents: list[Document],
    file_path: Path,
) -> list[Document]:
    """Add source information to each extracted document."""

    source_metadata = DOCUMENT_REGISTRY.get(
        file_path.name,
        {
            "title": file_path.stem.replace("_", " ").title(),
            "publisher": "Unknown",
            "mission": "Unknown",
            "document_type": "unknown",
            "url": "",
            "language": "unknown",
        },
    )

    enriched_documents = []

    for document in documents:
        document.metadata.update(source_metadata)

        document.metadata["source"] = file_path.name
        document.metadata["local_path"] = str(file_path)
        document.metadata["ingestion_date"] = date.today().isoformat()

        enriched_documents.append(document)

    return enriched_documents

In [ ]:
def ingest_directory(directory: Path) -> list[Document]:
    """Load and enrich all supported documents in a directory."""

    supported_extensions = {".pdf", ".txt", ".md"}

    all_documents = []

    for file_path in sorted(directory.rglob("*")):
        if not file_path.is_file():
            continue

        if file_path.suffix.lower() not in supported_extensions:
            continue

        print(f"Loading: {file_path.name}")

        try:
            documents = load_document(file_path)
            documents = enrich_metadata(documents, file_path)

            all_documents.extend(documents)

            print(f"Extracted {len(documents)} document units.")

        except Exception as error:
            print(f"Failed to load {file_path.name}: {error}")

    return all_documents

In [ ]:
# accessing metadata
all_documents[0].metadata

{'producer': 'Adobe PDF Library 19.21.79',
 'creator': 'Acrobat PDFMaker 19 for Word',
 'creationdate': '2019-11-19T10:06:32-06:00',
 'author': 'aaker',
 'company': 'EROS Data Center',
 'moddate': '2019-11-26T15:30:49-06:00',
 'sourcemodified': 'D:20191119160605',
 'title': 'Landsat 8 Data Users Handbook',
 'source': 'landsat_8_data_users_handbook.pdf',
 'total_pages': 114,
 'page': 0,
 'page_label': '1',
 'publisher': 'USGS',
 'mission': 'Landsat 8',
 'document_type': 'technical_handbook',
 'url': 'https://www.usgs.gov/media/files/landsat-8-data-users-handbook',
 'language': 'en',
 'local_path': '/content/drive/MyDrive/Satellite_Intelligence_Assistant/data/raw/landsat/landsat_8_data_users_handbook.pdf',
 'ingestion_date': '2026-11-09'}

In [ ]:
documents

In [ ]:
landsat_documents = ingest_directory(LANDSAT_DIR)
sentinel2_documents = ingest_directory(SENTINEL2_DIR)

all_documents = landsat_documents + sentinel2_documents

print(f"Total extracted document units: {len(all_documents)}")

Loading: landsat_8_data_users_handbook.pdf
Extracted 114 document units.
Loading: landsat_9_data_users_handbook.pdf
Extracted 115 document units.
Loading: sentinel_2_documentation.pdf
Extracted 40 document units.
Total extracted document units: 269


In [ ]:
def validate_documents(
    documents: list[Document],
) -> dict[str, Any]:
    """Check the basic quality of extracted documents.
    Ensuring document is readable and information may be extracted."""

    total_documents = len(documents)

    empty_documents = [
        document
        for document in documents
        if not document.page_content.strip()
    ]

    missing_source = [
        document
        for document in documents
        if not document.metadata.get("source")
    ]

    short_documents = [
        document
        for document in documents
        if len(document.page_content.strip()) < 50
    ]

    report = {
        "total_documents": total_documents, # nb of pages
        "empty_documents": len(empty_documents),
        "missing_source": len(missing_source),
        "short_documents": len(short_documents),
    }

    return report

In [ ]:
validation_report = validate_documents(all_documents)

for metric, value in validation_report.items():
    print(f"{metric}: {value}")

total_documents: 269
empty_documents: 0
missing_source: 0
short_documents: 0


In [ ]:
import json

OUTPUT_FILE = PROCESSED_DATA_DIR / "ingested_documents.jsonl"
# To avoid having to reread and reload all the PDFs at each execution,
# we save the ingestion result in JSONL format in the processed subfolder
# JSONL means JSON Lines: each line is an independent JSON object.

with OUTPUT_FILE.open("w", encoding="utf-8") as file:
    for document in all_documents:
        record = {
            "page_content": document.page_content,
            "metadata": document.metadata,
        }
        file.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )
print(f"Saved {len(all_documents)} documents to {OUTPUT_FILE}")

Saved 269 documents to /content/drive/MyDrive/Satellite_Intelligence_Assistant/data/processed/ingested_documents.jsonl


In [ ]:
reloaded_documents = []
with OUTPUT_FILE.open("r", encoding="utf-8") as file:
    for line in file:
        record = json.loads(line)

        document = Document(
            page_content=record["page_content"],
            metadata=record["metadata"],
        )
        reloaded_documents.append(document)
print(f"Reloaded documents: {len(reloaded_documents)}")

Reloaded documents: 269


In [ ]:
reloaded_documents

[Document(metadata={'producer': 'Adobe PDF Library 19.21.79', 'creator': 'Acrobat PDFMaker 19 for Word', 'creationdate': '2019-11-19T10:06:32-06:00', 'author': 'aaker', 'company': 'EROS Data Center', 'moddate': '2019-11-26T15:30:49-06:00', 'sourcemodified': 'D:20191119160605', 'title': 'Landsat 8 Data Users Handbook', 'source': 'landsat_8_data_users_handbook.pdf', 'total_pages': 114, 'page': 0, 'page_label': '1', 'publisher': 'USGS', 'mission': 'Landsat 8', 'document_type': 'technical_handbook', 'url': 'https://www.usgs.gov/media/files/landsat-8-data-users-handbook', 'language': 'en', 'local_path': '/content/drive/MyDrive/Satellite_Intelligence_Assistant/data/raw/landsat/landsat_8_data_users_handbook.pdf', 'ingestion_date': '2026-11-09'}, page_content='LSDS-1574 \nVersion 5.0 \nDepartment of the Interior \nU.S. Geological Survey \n \n \n \n \n \nLandsat 8 (L8) \nData Users Handbook \n \n \n \n \n \n \n \n \n \n \n \n \n \nVersion 5.0 \n \nNovember 2019'),
 Document(metadata={'producer'